In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat

In [ ]:
PATH_OR = os.path.join('..', 'content', 'mnist_original.mat')
PATH_AD = os.path.join('..', 'content', 'adversarial_data_1000.mat')

In [3]:
TRAIN_RATIO = 0.9
CLUSTERS = 856
EPOCHS = 10

In [4]:
mnist = loadmat(PATH_OR)
adverserial = loadmat(PATH_AD)

In [5]:
mnist_data = mnist['data'].T
mnist_labels = mnist['label'].reshape(-1)

adversarial_data = adverserial['adversarial_examples']
adversarial_labels = adverserial['labels'].reshape(-1)

In [6]:
if mnist_data.dtype != adversarial_data.dtype:
    mnist_data = mnist_data.astype(adversarial_data.dtype)

combined_data = np.concatenate((mnist_data, adversarial_data), axis=0)
combined_labels = np.concatenate((mnist_labels, adversarial_labels), axis=0)

In [7]:
train_mnist_data, test_mnist_data = mnist_data[:int(len(mnist_data) * TRAIN_RATIO)], mnist_data[int(len(mnist_data) * TRAIN_RATIO):]
train_mnist_labels, test_mnist_labels = mnist_labels[:int(len(mnist_labels) * TRAIN_RATIO)], mnist_labels[int(len(mnist_labels) * TRAIN_RATIO):]

train_adversarial_data, test_adversarial_data = adversarial_data[:int(len(adversarial_data) * TRAIN_RATIO)], adversarial_data[int(len(adversarial_data) * TRAIN_RATIO):]
train_adversarial_labels, test_adversarial_labels = adversarial_labels[:int(len(adversarial_labels) * TRAIN_RATIO)], adversarial_labels[int(len(adversarial_labels) * TRAIN_RATIO):]

train_combined_data, test_combined_data = np.concatenate((train_mnist_data, train_adversarial_data), axis=0), np.concatenate((test_mnist_data, test_adversarial_data), axis=0)
train_combined_labels, test_combined_labels = np.concatenate((train_mnist_labels, train_adversarial_labels), axis=0), np.concatenate((test_mnist_labels, test_adversarial_labels), axis=0)

In [8]:
from sklearn.cluster import KMeans
from scipy.stats import mode

# Normal Training

In [9]:
kmeans_or = KMeans(n_clusters=CLUSTERS, n_init="auto", max_iter=EPOCHS, random_state=42)
kmeans_or.fit(train_mnist_data)
cluster_assignments = kmeans_or.labels_

In [10]:
cluster_labels_or = np.zeros(CLUSTERS, dtype=int)
for i in range(CLUSTERS):
    mask = cluster_assignments == i
    cluster_labels_or[i] = mode(train_mnist_labels[mask], keepdims=True).mode[0]

In [11]:
pred_test_mnist_label_or = cluster_labels_or[kmeans_or.predict(test_mnist_data)]
accuracy = np.mean(pred_test_mnist_label_or == test_mnist_labels)
print(f"Original Accuracy: {accuracy * 100:.2f}%")

pred_test_adverserial_label_or = cluster_labels_or[kmeans_or.predict(adversarial_data)]
accuracy = np.mean(pred_test_adverserial_label_or == adversarial_labels)
print(f"Adverserial Accuracy: {accuracy * 100:.2f}%")

pred_test_combined_label_or = cluster_labels_or[kmeans_or.predict(test_combined_data)]
accuracy = np.mean(pred_test_combined_label_or == test_combined_labels)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")

Original Accuracy: 92.37%
Adverserial Accuracy: 9.70%
Overall Accuracy: 91.20%


# Adverserial Training

In [12]:
kmeans_ad = KMeans(n_clusters=CLUSTERS, n_init="auto", max_iter=EPOCHS, random_state=42)
kmeans_ad.fit(train_combined_data)
cluster_assignments_ad = kmeans_ad.labels_

In [13]:
cluster_labels_ad = np.full(CLUSTERS, -1, dtype=int)
for i in range(CLUSTERS):
    mask = cluster_assignments_ad == i
    if not np.any(mask):
        continue
    cluster_labels_ad[i] = mode(train_combined_labels[mask], keepdims=True).mode[0]

In [14]:
pred_test_mnist_label_ad = cluster_labels_ad[kmeans_ad.predict(test_mnist_data)]
accuracy = np.mean(pred_test_mnist_label_ad == test_mnist_labels)
print(f"Original Accuracy: {accuracy * 100:.2f}%")

pred_test_adverserial_label_ad = cluster_labels_ad[kmeans_ad.predict(test_adversarial_data)]
accuracy = np.mean(pred_test_adverserial_label_ad == test_adversarial_labels)
print(f"Adverserial Accuracy: {accuracy * 100:.2f}%")

pred_test_combined_label_ad = cluster_labels_ad[kmeans_ad.predict(test_combined_data)]
accuracy = np.mean(pred_test_combined_label_ad == test_combined_labels)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")

Original Accuracy: 92.51%
Adverserial Accuracy: 10.00%
Overall Accuracy: 91.35%


# Analysis

In [15]:
if 'kmeans_ad' in globals():
    cluster_assignments_full = kmeans_ad.predict(combined_data)
else:
    cluster_assignments_full = kmeans_or.predict(combined_data)

df_analysis = pd.DataFrame({
    'original_label': combined_labels,
    'is_adversarial': [False] * mnist_data.shape[0] + [True] * adversarial_data.shape[0],
    'cluster_label': cluster_assignments_full
})

cluster_analysis = df_analysis.groupby('cluster_label')['is_adversarial'].agg(['count', 'sum'])
cluster_analysis.rename(columns={'count': 'total_in_cluster', 'sum': 'adversarial_in_cluster'}, inplace=True)
cluster_analysis['original_in_cluster'] = cluster_analysis['total_in_cluster'] - cluster_analysis['adversarial_in_cluster']

cluster_analysis['percentage_adversarial'] = (cluster_analysis['adversarial_in_cluster'] / cluster_analysis['total_in_cluster']) * 100
cluster_analysis['percentage_original'] = (cluster_analysis['original_in_cluster'] / cluster_analysis['total_in_cluster']) * 100

display(cluster_analysis.where(cluster_analysis['total_in_cluster'] > 0).sort_values(by='percentage_adversarial', ascending=False).head(5))
display(cluster_analysis.where(cluster_analysis['total_in_cluster'] > 0).sort_values(by='percentage_original', ascending=False).head(5))


,total_in_cluster,adversarial_in_cluster,original_in_cluster,percentage_adversarial,percentage_original
cluster_label,,,,,
60,1008,1000,8,99.206349,0.793651
840,49,0,49,0.000000,100.000000
839,82,0,82,0.000000,100.000000
8,105,0,105,0.000000,100.000000
855,63,0,63,0.000000,100.000000


,total_in_cluster,adversarial_in_cluster,original_in_cluster,percentage_adversarial,percentage_original
cluster_label,,,,,
855,63,0,63,0.0,100.0
0,33,0,33,0.0,100.0
1,197,0,197,0.0,100.0
2,53,0,53,0.0,100.0
3,64,0,64,0.0,100.0


In [16]:
pure_original_clusters = cluster_analysis[cluster_analysis['percentage_original'] == 100]
pure_adversarial_clusters = cluster_analysis[cluster_analysis['percentage_adversarial'] == 100]
mixed_clusters = cluster_analysis[(cluster_analysis['percentage_original'] > 0) & (cluster_analysis['percentage_adversarial'] > 0)]

print(f"Number of clusters with only original examples: {len(pure_original_clusters)}")
print(f"Number of clusters with only adversarial examples: {len(pure_adversarial_clusters)}")
print(f"Number of clusters with mixed examples: {len(mixed_clusters)}")

if not mixed_clusters.empty:
    print("\nExamples of mixed clusters:")
    display(mixed_clusters.head())

Number of clusters with only original examples: 855
Number of clusters with only adversarial examples: 0
Number of clusters with mixed examples: 1

Examples of mixed clusters:


,total_in_cluster,adversarial_in_cluster,original_in_cluster,percentage_adversarial,percentage_original
cluster_label,,,,,
60,1008,1000,8,99.206349,0.793651


In [17]:
print("Effectiveness of k-means clustering in separating original and adversarial examples:")
print("Based on the cluster analysis:")
print(f"- Number of clusters with only original examples: {len(pure_original_clusters)}")
print(f"- Number of clusters with only adversarial examples: {len(pure_adversarial_clusters)}")
print(f"- Number of clusters with mixed examples: {len(mixed_clusters)}")

if not pure_adversarial_clusters.empty:
    print(f"\nWhile there are {len(pure_original_clusters)} clusters containing only original examples, indicating good separation for the majority of the original data, there are {len(pure_adversarial_clusters)} clusters containing only adversarial examples, suggesting some level of separation of adversarial data as well.")
elif not mixed_clusters.empty:
     print(f"\nWhile there are {len(pure_original_clusters)} clusters containing only original examples, indicating good separation for the majority of the original data, there are no clusters containing only adversarial examples. Instead, the adversarial examples are concentrated in {len(mixed_clusters)} mixed cluster(s).")
else:
    print("\nAll clusters containing adversarial examples are mixed with original examples.")

Effectiveness of k-means clustering in separating original and adversarial examples:
Based on the cluster analysis:
- Number of clusters with only original examples: 855
- Number of clusters with only adversarial examples: 0
- Number of clusters with mixed examples: 1

While there are 855 clusters containing only original examples, indicating good separation for the majority of the original data, there are no clusters containing only adversarial examples. Instead, the adversarial examples are concentrated in 1 mixed cluster(s).


Implications for adversarial training or detection:
The concentration of adversarial examples in mixed clusters suggests that while k-means didn't create pure adversarial clusters, these mixed clusters represent regions in the feature space where original and adversarial examples are close.
For adversarial training, these mixed clusters could be valuable for identifying boundary-proximal examples. Training on data from these clusters might help a model become more robust to perturbations that move original examples close to the adversarial region.
For adversarial detection, analyzing the characteristics of the mixed clusters could help in building detectors that are sensitive to the subtle differences between original and adversarial examples in these ambiguous regions.

Key takeaways:
K-means clustering with 856 clusters resulted in a large number of clusters containing only original examples. However, adversarial examples did not form pure clusters and were instead concentrated in a small number of mixed clusters.

Overall, while k-means clustering provided some insights into the distribution of original and adversarial examples, its effectiveness in completely separating them is limited based on this analysis.